# Feature Engineering for Credit Risk Modeling

This notebook transforms raw Xente transaction data into customer-level
behavioral features. All logic lives in `src/data_processing.py` — this
notebook only calls it and inspects the results.

In [1]:
import sys; sys.path.append("..")
import pandas as pd

from src.data_processing import (
    load_raw_data,
    compute_customer_features,
    NUMERICAL_COLS,
    CATEGORICAL_COLS,
)

## Load raw transaction data

In [2]:
df = load_raw_data("../data/raw/alternate_data.csv")
df.head()

2026-09-01 16:24:15,710 [INFO] Loading raw data from: ../data/raw/alternate_data.csv
2026-09-01 16:24:16,179 [INFO] Loaded 95662 transactions, 3742 unique customers.


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15 02:18:49+00:00,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15 02:19:08+00:00,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15 02:44:21+00:00,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15 03:32:55+00:00,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15 03:34:21+00:00,2,0


## Customer-level aggregation

Collapses transaction-level rows into one row per `CustomerId`:
- **Aggregates**: total/avg/std transaction amount, transaction count, total value
- **Velocity**: transactions per day, value per day
- **Temporal**: mean transaction hour/day
- **Diversity**: unique products/categories/providers
- **Channel behavior**: transaction counts per channel

In [8]:
# this notebook is just for feature engineering understanding we don't need to 
# remove the RFM features yet (to avoid leakage, we're supposed to do this before giving
# the data to the model, so we will do this in the next notebook), for now we will just
# save the data WITH the RFM features as customer_features_rfm.csv

customer_features = compute_customer_features(df)
customer_features.to_csv("../data/processed/customer_features_rfm.csv", index=False)
print(f"Shape: {customer_features.shape}")
customer_features.head()

2026-09-01 17:13:01,187 [INFO] Computing customer features with snapshot date: 2019-02-14 10:01:28+00:00
2026-09-01 17:13:02,984 [INFO] Customer feature assembly complete: 3742 customers


Shape: (3742, 32)


,CustomerId,total_amount,avg_amount,std_amount,transaction_count,total_value,fraud_count,first_txn,last_txn,txn_hour_mean,...,amount_cv,preferred_hour,weekend_ratio,unique_products,unique_categories,unique_providers,ChannelId_1,ChannelId_2,ChannelId_3,ChannelId_5
0,CustomerId_1,-10000.0,-10000.000000,0.000000,1,10000,0,2018-11-21 16:49:14+00:00,2018-11-21 16:49:14+00:00,16.000000,...,0.000000,16,0.000000,1,1,1,0,1,0,0
1,CustomerId_10,-10000.0,-10000.000000,0.000000,1,10000,0,2018-11-21 16:49:09+00:00,2018-11-21 16:49:09+00:00,16.000000,...,0.000000,16,0.000000,1,1,1,0,1,0,0
2,CustomerId_1001,20000.0,4000.000000,6558.963333,5,30400,0,2018-11-16 07:53:19+00:00,2018-11-16 08:20:39+00:00,7.800000,...,1.639741,8,0.000000,3,2,3,0,2,3,0
3,CustomerId_1002,4225.0,384.090909,560.498966,11,4775,0,2018-11-15 18:50:09+00:00,2019-01-18 10:05:00+00:00,13.454545,...,1.459287,14,0.181818,3,2,2,0,6,5,0
4,CustomerId_1003,20000.0,3333.333333,6030.478146,6,32000,0,2019-02-01 14:58:07+00:00,2019-02-01 15:04:51+00:00,14.333333,...,1.809143,14,0.000000,4,2,3,0,2,4,0


In [4]:
customer_features[NUMERICAL_COLS].describe().T

,count,mean,std,min,25%,50%,75%,max
total_amount,3742.0,171737.740647,2.717305e+06,-1.049000e+08,4077.437500,20000.000000,79967.750000,8.345124e+07
avg_amount,3742.0,15715.616012,1.676991e+05,-4.250000e+05,1000.000000,2583.846429,4877.613636,8.601821e+06
std_amount,3742.0,13605.166177,9.689344e+04,0.000000e+00,501.141058,3184.898145,6745.368782,3.309916e+06
transaction_count,3742.0,25.564404,9.692960e+01,1.000000e+00,2.000000,7.000000,20.000000,4.091000e+03
total_value,3742.0,253102.528327,2.715877e+06,5.000000e+01,6500.000000,32000.000000,102060.000000,1.049000e+08
recency_days,3742.0,31.461251,2.711893e+01,1.000000e+00,6.000000,25.000000,54.000000,9.100000e+01
fraud_count,3742.0,0.051577,7.743284e-01,0.000000e+00,0.000000,0.000000,0.000000,3.100000e+01
txn_hour_mean,3742.0,12.560712,3.894273e+00,0.000000e+00,10.000000,12.500000,15.000000,2.300000e+01
txn_day_mean,3742.0,15.652035,6.941424e+00,1.000000e+00,11.000000,15.848848,20.250000,3.100000e+01
txns_per_day,3742.0,0.611913,1.874089e+00,1.098901e-02,0.058140,0.197667,0.613588,4.596629e+01
